# LOAD DATA

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from Bio import pairwise2
from pathlib import Path
DATA_PATH = Path("../../../../data/bpRNA/")
df = pd.read_csv(DATA_PATH / "bpRNA.csv", index_col="id") 

# Similarity functions by chunks

In [ ]:
# keep only nucleotides ACGU
def valid_sequence_ids(df):
    valid_ids = []
    for i in range(len(df)):
        save = True
        for l in df.iloc[i].sequence:
            if l not in ["A", "C", "G", "U"]: 
                save = False
        if save:
            valid_ids.append(df.iloc[i].name)
    print(f"Total IDs: {len(df)} , Valid IDs: {len(valid_ids)}")
    df_valid = df.loc[valid_ids]
    return df_valid

## compute similarity between two sequences
def get_match_dict(symbols, Match, Mismatch): 
    match_dic = {}
    for simbolA in symbols:
        for simbolB in symbols:
            if simbolA==simbolB:
                match_dic[(simbolA, simbolB)] = Match
            else:
                match_dic[(simbolA, simbolB)] = Mismatch
    return match_dic
    
def get_sim_conf():
    Match = 1
    Mismatch = -2
    GapOpen=-5
    GapExtend=-2
    symbols=["A","C","G","U"]
    match_dic = get_match_dict(symbols, Match, Mismatch)
    sim_conf = match_dic, GapOpen, GapExtend
    return sim_conf

def compute_similarity(col_seq, row_seq, sim_conf):
    alignm = pairwise2.align.localds(col_seq, row_seq, sim_conf[0], sim_conf[1], sim_conf[2], one_alignment_only=True)[0]
    max_len = max(len(alignm.seqB),len(alignm.seqA))
    IDscore = sum(a==b for a,b in zip(alignm.seqA, alignm.seqB)) / max_len
    return IDscore


# execute comparisons for a slice of the dataframe
def expected_num_comparisons(n, start, end):
    return sum(n - i - 1 for i in range(start, end))

def compute_slice(df, start, end):
    ids = df.index.to_list()
    n = len(df)
    # IDs of the rows to compute similarity for
    row_ids = ids[start:end]
    
    sims = pd.DataFrame(
        np.zeros((len(row_ids), n), dtype=np.float32),
        index=row_ids,
        columns=ids,
    )

    total = expected_num_comparisons(n, start, end)
    sim_conf = get_sim_conf()

    with tqdm(total=total, desc=f"Rows {start}:{end}") as pbar:
        for i in range(start, end):
            row_id = ids[i]
            seq_i = df.loc[row_id, "sequence"]

            for j in range(i + 1, n):
                col_id = ids[j]
                seq_j = df.loc[col_id, "sequence"]

                sim = compute_similarity(seq_i, seq_j, sim_conf)

                sims.loc[row_id, col_id] = sim

                pbar.update(1)

    return sims


# Compute by chunks (Upper triangle)

In [ ]:
df = valid_sequence_ids(df)
chunks = 17
n = len(df)
start = 0
end = n
print(f"Expected number of total comparisons: {expected_num_comparisons(n, start, end)}")

acc = np.cumsum(n - np.arange(n) - 1)
parts = [acc.max() * i  / chunks for i in range(1, chunks + 1)]
equal_ids = [0] + [np.searchsorted(acc, part).item() for part in parts]
equal_ids[-1] = n
computations = []
for part in equal_ids[1:]:
    end = part
    computations.append(expected_num_comparisons(n, start, end))
    start = end
print(f"Average computations per chunk: {int(sum(computations) / len(computations))}")  


In [ ]:
TMP_PATH = DATA_PATH / "distances" / "chunks"  # same folder as seqsim.py
TMP_PATH.mkdir(parents=True, exist_ok=True)

# This loop executes the similarity computations for all chunks and saves the results to HDF5 files.
# For a faster computation yo can parallel this procees with the selected ids
for chunk_id in range(len(equal_ids) - 1):
    start = equal_ids[chunk_id]
    end = equal_ids[chunk_id + 1]
    out = compute_slice(df, start, end)
    out_path = TMP_PATH / f"bprna_seqsim_{end}.h5"
    out.to_hdf(
    out_path,
    key="seqsim",
    mode="w",
    format="fixed",
)


In [ ]:
# Reading the computed chunks and concatenating them into a full similarity matrix
parts = []
for chunk in equal_ids[1:]:
    file_path = TMP_PATH / f"bprna_seqsim_{str(chunk)}.h5"
    parts.append(pd.read_hdf(file_path, key="seqsim"))

seqsim = pd.concat(parts, axis=0)

arr = seqsim.to_numpy(dtype=np.float32)
arr = arr + arr.T
np.fill_diagonal(arr, 1.0) # similarity of a sequence with itself is 1.0

seqsim_full = pd.DataFrame(arr, index=seqsim.index, columns=seqsim.columns, dtype=np.float32)
seqsim_full.to_hdf(DATA_PATH / "distances" / "bprna_seqsim_full_ACGU.h5", key="seqsim", mode="w", format="fixed")